# ProductIQ Phase 2 — Step 2.8: Product Attribute Engineering

Deterministic, profile-driven extraction of structured product attributes from canonical metadata (primarily `description`).
No LLMs, embeddings, or probabilistic models — controlled vocabularies and boundary-aware regex only.

## 1. Phase objective

Workflow: **canonical dataframe → vocabulary profiling → controlled rules → deterministic extraction → attribute-enriched dataframe**.

Rules:

- Add attribute columns only; do not mutate canonical fields.
- Preserve row count, order, and `product_id`.
- Multi-valued attributes use the `|` delimiter (sorted, deduplicated).
- No match → pandas `<NA>` (not `"unknown"`).

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from data.attributes import (
    ATTRIBUTE_COLUMN_ORDER,
    ATTRIBUTE_DEFINITIONS,
    ATTRIBUTE_VOCABULARY_RULES,
    MULTI_VALUE_DELIMITER,
    ProductAttributeExtractor,
    ProductAttributeVocabularyProfiler,
)
from data.catalog import AjioCanonicalProductBuilder, CANONICAL_COLUMN_ORDER
from data.deduplication import AjioDataDeduplicator
from data.ingestion import AjioDataLoader
from data.preprocessing import AjioDataCleaner, AjioDataNormalizer
from data.validation import AjioDataValidator


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_PATH = PROJECT_ROOT / "resources" / "raw" / "Ajio Fashion Clothing.csv"

raw_df = AjioDataLoader().load(DATASET_PATH)
AjioDataValidator().validate(raw_df)
cleaned_df = AjioDataCleaner().clean(raw_df).dataframe
normalized_df = AjioDataNormalizer().normalize(cleaned_df).dataframe
prepared_df = AjioDataDeduplicator().deduplicate(normalized_df).dataframe
canonical_df = AjioCanonicalProductBuilder().build(prepared_df).dataframe

print("Pipeline row count:", len(canonical_df))
print("Canonical columns:", len(CANONICAL_COLUMN_ORDER))

## 2. Dataset description profiling

Brief look at the text field used for attribute matching.

In [ ]:
description_lengths = canonical_df["description"].astype("string").str.len()
print(description_lengths.describe())
canonical_df["description"].head(5)

## 3. Attribute vocabulary profiling

Read-only counts for each controlled-vocabulary expression (same rules as extraction).

In [ ]:
vocab_profile = ProductAttributeVocabularyProfiler().profile(canonical_df)
print("Profiled rows:", vocab_profile.total_rows)
print("Expression summaries:", len(vocab_profile.term_summaries))

## 4. Review candidate vocabulary

Top canonical values per attribute (by product row coverage).

In [ ]:
rows = []
for attribute in ATTRIBUTE_COLUMN_ORDER:
    summaries = vocab_profile.summaries_for_attribute(attribute)
    by_value: dict[str, int] = {}
    for summary in summaries:
        by_value[summary.canonical_value] = max(
            by_value.get(summary.canonical_value, 0), summary.row_count
        )
    for value, count in sorted(by_value.items(), key=lambda item: item[1], reverse=True)[:8]:
        rows.append(
            {
                "attribute": attribute,
                "canonical_value": value,
                "row_count": count,
                "coverage_pct": round(count / vocab_profile.total_rows * 100, 2),
            }
        )

pd.DataFrame(rows)

## 5. Define / inspect controlled rules

Vocabularies live in `data.attributes.rules`. Longer phrases are ordered before shorter overlaps (e.g. `t-shirt` before `shirt`, `patch pocket` before `pocket`).

In [ ]:
rule_rows = [
    {
        "attribute": rule.attribute,
        "canonical_value": rule.canonical_value,
        "expressions": ", ".join(rule.expressions),
    }
    for rule in ATTRIBUTE_VOCABULARY_RULES
]
pd.DataFrame(rule_rows).head(20)

In [ ]:
pd.DataFrame(
    [
        {
            "name": definition.name,
            "multi_valued": definition.multi_valued,
            "dtype": definition.dtype,
            "delimiter": MULTI_VALUE_DELIMITER if definition.multi_valued else pd.NA,
        }
        for definition in ATTRIBUTE_DEFINITIONS
    ]
)

## 6. Run deterministic extraction

In [ ]:
extraction_result = ProductAttributeExtractor().extract(canonical_df)
enriched_df = extraction_result.dataframe
print(extraction_result.report.summary())

## 7. Inspect enriched examples

In [ ]:
sample_cols = ["product_id", "description", *ATTRIBUTE_COLUMN_ORDER]
enriched_df.loc[
    enriched_df["description"].astype("string").str.contains("Striped", case=False, na=False),
    sample_cols,
].head(3)

## 8. Attribute coverage statistics

In [ ]:
coverage_rows = []
report = extraction_result.report
for attribute in ATTRIBUTE_COLUMN_ORDER:
    matched = report.matched_row_counts[attribute]
    coverage_rows.append(
        {
            "attribute": attribute,
            "matched_rows": matched,
            "unmatched_rows": report.unmatched_row_counts[attribute],
            "coverage_pct": round(matched / report.output_row_count * 100, 2),
        }
    )
pd.DataFrame(coverage_rows)

## 9. Preservation checks

In [ ]:
print("Row count:", len(enriched_df) == len(canonical_df))
print("Index equal:", enriched_df.index.equals(canonical_df.index))
print("product_id equal:", enriched_df["product_id"].equals(canonical_df["product_id"]))
canonical_unchanged = all(
    enriched_df[column].equals(canonical_df[column]) for column in CANONICAL_COLUMN_ORDER
)
print("Canonical columns unchanged:", canonical_unchanged)
print("Input canonical not mutated:", canonical_df.columns.tolist() == list(CANONICAL_COLUMN_ORDER))

## 10. Final schema

In [ ]:
print("Total columns:", len(enriched_df.columns))
print("Canonical:", list(CANONICAL_COLUMN_ORDER))
print("Attributes:", list(ATTRIBUTE_COLUMN_ORDER))
enriched_df.dtypes

## 11. Phase 2.8 summary

- Profiling on AJIO descriptions justified all eight candidate attribute columns (`product_type`, `fit`, `pattern`, `sleeve`, `neckline`, `material`, `product_features`, `style_attributes`).
- Extraction is deterministic, case-insensitive, whitespace-normalized, and phrase/boundary aware.
- `product_type` and `pattern` have the highest match rates; `fit`, `sleeve`, and `material` are sparser because many titles omit those tokens.
- Multi-valued fields use sorted unique values joined with `|`.
- Next phases may add ML or catalog joins; this step stays rule-based and auditable.